# Dynamische Konzepte der OOP

## Objekte sind dynamisch

* Unsere bisherigen selbstdefinierten Klassen ware vorwiegend statisch programmiert
    * Im Konstruktor werden die Attribute eines Objekts definiert
* Objekte aber auch zur Laufzeit jederzeit neue Attribute und Methoden gesetzt bekommen
    * Die Kontrolle hierüber übernimmt die Anwendung
    * Die dynamischen Attribute beziehen sich auf jede Instanz

In [ ]:
class Empty:
    pass

instance1 = Empty()
instance2 = Empty()

instance1.name = 'Hugo'
instance2.number = 9

print(instance1.name)
print(instance2.number(5))

Hugo
14


* Es können allerdings auch Methoden hinzugefügt werden
    * Vorsicht: Dies ist nicht ganz so einfach, da Methoden ja als ersten Parameter die Referenz `self` entgegennehmen müssen
    * Weiterhin sollte der Interpreter diesen Parameter automatisch setzen, sonst sieht der Aufruf sehr ungewöhnlich aus
        * Hierzu dient der Typ `MethodType`

In [5]:
def add(object, number):
    object.number += number
    return object.number
instance2.strange_add = add
print(instance2.strange_add(instance2, 5))

from types import MethodType
instance2.add = MethodType(add, instance2)
print(instance2.add(5))


29
34


* Sollen eine Klasse dynamisch erweitert werden, ist das sogar noch einfacher, da nun dem Python-Klassenobjekt einfach eine Funktion zugewiesen wird
    * Diese neue Funktion ist dann für alle Instanzen aufrufbar

In [6]:
Empty.demo = lambda self: 'Hello'
print(instance1.demo())
print(instance2.demo())


Hello
Hello


## Dynamische Klassen

* Auch Klassen können dynamisch Attribute generieren, statt sie statisch anzulegen

In [ ]:
class Dynamic:
    def __getattribute__(self, name):
        return f'i have an attribute named {name}'
    def __setattr__(self, name, value):
        print(f'i am pretending to set value {value} to attribute {name}')

dynamic = Dynamic()
print(dynamic.foo)
print(dynamic.goo)

dynamic.foo = 42
print(dynamic.foo)


i have an attribute named foo
i have an attribute named goo
i am pretending to set value42 to attribute foo
i have an attribute named foo


* das gilt auch für Methoden

In [29]:
class Dynamic:
    def __getattribute__(self, name):
        def what_you_want(*varargs, ** kwargs):
            print(f'calling what_you_want with varargs {varargs} and kwargs {kwargs}')
            return 'i can do anything!'
        if name.startswith('m_'):
            return what_you_want
        else:
            return f'i have an attribute named {name}'
    def __setattr__(self, name, value):
        print(f'i am pretending to set value {value} to attribute {name}')
dynamic = Dynamic()

print(dynamic.m_answer())
print(dynamic.m_answer(42, 3, 7))


calling what_you_want with varargs () and kwargs {}
i can do anything!
calling what_you_want with varargs (42, 3, 7) and kwargs {}
i can do anything!


## Aspektorientierte Programmierung

### Allgemeines
* Ein Aspekt erweitert eine existierende Funktionalität, ohne dass dessen Implementierung geändert werden muss
* Die Aspektlogik ist nicht typisch für **eine** Funktionalität, sondern ist allgemein gültig
    * In der Modellierung spricht man von sogenannten Querschnittsfunktionen / cross cutting concerns

### Umsetzung in Python
* Die Umsetzung von AOP ist im Gegensatz zu statisch typisierten Sprachen wie Java sehr einfach und bereits in der elementaren Sprachspezifikation enthalten
    * Jede Funktionalität ist Bestandteil einer Funktion, die zur Laufzeit problemlos ausgetauscht werden kann
        * "Monkey Patching"
    * Die Allgemeine Lösung einer AOP-Logik ist etwas komplexer

In [31]:
def around(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Finished {func.__name__}")
        return result
    return wrapper
def plus(number1, number2):
    return number1 + number2

print(plus(1,2))
plus = around(plus)
print(plus(1,2))


3
Calling plus
Finished plus
3


* Monkey Patching ist für Anwendungen etwas intransparent und damit fehleranfällig
    * Jede Funktion kann irgendwo zur Ausführungszeit komplett geändert werden
    * Damit außerhalb von Test- und Entwicklungsphasen in der Software-Entwicklung nicht genutzt
* Bessere Variante: Python Decorators
    * Die oben geschriebene Aspektfunktion `around` ist sofort auch als Decorator nutzbar
    * Allerdings wird die Nutzung des Aspekts bei der Funktionsdefinition mit angegeben
        * und damit ist die Wartbarkeit und Übersichtlichkeit gegeben

In [33]:
@around
def minus(number1, number2):
    return number1 - number2

print(minus(1,2))

Calling minus
Finished minus
-1


* Falls die Aspektlogik selber konfiguriert werden soll, muss diese Parameter entgegennehmen können
    * Paramterisierte Decorators
    * Die Umsetzung nutzt eine weitere Verschachtelung in der Aspektfunktion
    

In [35]:
def parameterized_around(detailed=False):
    def decorator(func):
        def wrapper(*args, **kwargs):
            if detailed:
                print(f"Calling {func.__name__} with varargs {args} and kwargs {kwargs}")
            else:
                print(f"Calling {func.__name__}")
            result = func(*args, **kwargs)
            print(f"Finished {func.__name__}")
            return result
        return wrapper
    return decorator

@parameterized_around()
def times(number1, number2):
    return number1 * number2

@parameterized_around(True)
def divide_by(number1, number2):
    return number1 / number2

print(times(2,5))
print(divide_by(2,5))


Calling times
Finished times
10
Calling divide_by with varargs (2, 5) and kwargs {}
Finished divide_by
0.4
